# 12.1 Graph representation & tasks

Representing a graph well is the first model choice in graph learning.

Graph learning begins where graph theory and linear algebra meet. The adjacency matrix stores relationships, the feature matrix stores node evidence, and labels can live on nodes, edges, or the whole graph.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 12
rng = np.random.default_rng(SEED)
random.seed(SEED)


def adjacency_from_graph(G):
    nodes = list(G.nodes())
    return nx.to_numpy_array(G, nodelist=nodes, dtype=float)


def labels_from_graph(G):
    labels = []
    for node in G.nodes():
        club = G.nodes[node].get("club", "Mr. Hi")
        value = 1 if club == "Officer" else 0
        labels.append(value)
    return np.array(labels, dtype=int)


def degree_features(A):
    degree = A.sum(axis=1)
    max_degree = max(1.0, float(degree.max()))
    normalized_degree = degree / max_degree
    clustering = np.array([nx.clustering(nx.from_numpy_array(A), i) for i in range(A.shape[0])])
    return np.column_stack([normalized_degree, clustering])


def make_sbm_graph(sizes, p_in, p_out, seed, feature_noise):
    blocks = len(sizes)
    probs = np.full((blocks, blocks), p_out, dtype=float)
    np.fill_diagonal(probs, p_in)
    G = nx.stochastic_block_model(sizes, probs, seed=seed)
    y = []
    for block_id, size in enumerate(sizes):
        y.extend([block_id] * size)
    y = np.array(y, dtype=int)
    base = np.eye(blocks, dtype=float)[y]
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(0.0, feature_noise, size=base.shape)
    X = base + noise
    return G, X, y


def make_graph_ladder(seed=SEED):
    ladder = []

    G1 = nx.Graph()
    G1.add_edges_from([(0, 1), (0, 2), (1, 2), (1, 3)])
    X1 = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.0, 0.0]])
    y1 = np.array([0, 0, 1, 1], dtype=int)
    ladder.append({"rung": "D1", "name": "4-node toy", "G": G1, "A": adjacency_from_graph(G1), "X": X1, "y": y1})

    G2 = nx.karate_club_graph()
    A2 = adjacency_from_graph(G2)
    X2 = degree_features(A2)
    y2 = labels_from_graph(G2)
    ladder.append({"rung": "D2", "name": "karate club", "G": G2, "A": A2, "X": X2, "y": y2})

    G3, X3, y3 = make_sbm_graph([18, 18], 0.34, 0.04, seed + 3, 0.28)
    ladder.append({"rung": "D3", "name": "synthetic SBM with noise", "G": G3, "A": adjacency_from_graph(G3), "X": X3, "y": y3})

    G4, X4, y4 = make_sbm_graph([30, 30, 30], 0.28, 0.07, seed + 4, 0.35)
    ladder.append({"rung": "D4", "name": "larger denser SBM", "G": G4, "A": adjacency_from_graph(G4), "X": X4, "y": y4})

    G5, X5, y5 = make_sbm_graph([35, 35, 35], 0.18, 0.11, seed + 5, 0.55)
    hub_edges = []
    for node in range(10, 95, 7):
        hub_edges.append((0, node))
    G5.add_edges_from(hub_edges)
    A5 = adjacency_from_graph(G5)
    ladder.append({"rung": "D5", "name": "noisy hub-heavy SBM", "G": G5, "A": A5, "X": X5, "y": y5})

    return ladder


def centroid_predict(Z, y):
    classes = np.unique(y)
    centroids = []
    for cls in classes:
        centroids.append(Z[y == cls].mean(axis=0))
    centroids = np.vstack(centroids)
    distances = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    indices = distances.argmin(axis=1)
    return classes[indices]


def node_accuracy(pred, y):
    return float(np.mean(np.asarray(pred) == np.asarray(y)))


def conductance(A, labels):
    labels = np.asarray(labels)
    group = labels == labels[0]
    if group.all() or (not group.any()):
        return 1.0
    cut = float(A[np.ix_(group, ~group)].sum())
    volume_a = float(A[group].sum())
    volume_b = float(A[~group].sum())
    denom = max(1e-12, min(volume_a, volume_b))
    return cut / denom


def normalized_adjacency(A):
    A_tilde = A + np.eye(A.shape[0])
    degree = A_tilde.sum(axis=1)
    inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-12)))
    return inv_sqrt @ A_tilde @ inv_sqrt


def plot_graph_panels(results, metric_name):
    cols = len(results)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 3))
    if cols == 1:
        axes = [axes]
    for ax, item in zip(axes, results):
        G = item["G"]
        colors = item["pred"]
        pos = nx.spring_layout(G, seed=SEED)
        nx.draw_networkx_nodes(G, pos, node_color=colors, cmap="viridis", node_size=80, ax=ax)
        nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.7, ax=ax)
        ax.set_title(f'{item["rung"]}: {item["metric"]:.3f}')
        ax.axis("off")
    plt.show()

    plt.figure(figsize=(6, 3))
    xs = [item["rung"] for item in results]
    ys = [item["metric"] for item in results]
    plt.plot(xs, ys, marker="o")
    plt.ylabel(metric_name)
    plt.xlabel("ladder rung")
    plt.title(f"{metric_name} vs rung")
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once (D1)

For a graph, $d=A\mathbf{1}$, node tasks use $X\in\mathbb{R}^{n\times d}$, edge tasks can use scores such as $(A^2)_{ij}$, and graph tasks can pool features such as $[|E|,\bar d,d_{\max}]$. The lesson toy should give $d=[2,3,2,1]$, $|E|=4$, $(A^2)_{2,3}=1$, and $[|E|,\bar d,d_{\max}]=[4,2,3]$.

In [ ]:

def build_graph_tensors(G, X):
    A = adjacency_from_graph(G)
    d = A.sum(axis=1).astype(int)
    edge_count = int(A.sum() // 2)
    common_neighbors = A @ A
    graph_features = np.array([edge_count, float(d.mean()), int(d.max())], dtype=float)
    task_views = {
        "node_shape": X.shape,
        "edge_candidates": np.argwhere(np.triu(1 - A, k=1) == 1),
        "graph_features": graph_features,
    }
    return A, X, d, common_neighbors, task_views

G_toy = nx.Graph()
G_toy.add_edges_from([(0, 1), (0, 2), (1, 2), (1, 3)])
X_toy = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.0, 0.0]])
A_toy, X_seen, d_toy, common_toy, views_toy = build_graph_tensors(G_toy, X_toy)

assert d_toy.tolist() == [2, 3, 2, 1]
assert int(A_toy.sum() // 2) == 4
assert int(common_toy[2, 3]) == 1
assert views_toy["graph_features"].tolist() == [4.0, 2.0, 3.0]
assert views_toy["node_shape"] == (4, 2)


The same tensors now expose node, edge, and graph views without changing the relationships. For D1, the upper triangle has exactly four edges, and the candidate non-edge list reminds us that zero can mean unknown rather than negative.

In [ ]:

print("A")
print(A_toy.astype(int))
print("degree", d_toy.tolist())
print("candidate non-edges", views_toy["edge_candidates"].tolist())
print("graph features", views_toy["graph_features"].tolist())


## 4. The dataset ladder

The F11 graph ladder is inline and CPU-only: D1 is a hand-checkable four-node toy, D2 is the bundled NetworkX karate club graph, D3 is a noisy stochastic block model, D4 is a larger/denser synthetic graph, and D5 is a harder noisy hub-heavy graph.

In [ ]:

ladder = make_graph_ladder()

for item in ladder:
    A = item["A"]
    X = item["X"]
    y = item["y"]
    edge_count = int(A.sum() // 2)
    classes = np.unique(y).tolist()
    print(item["rung"], item["name"])
    print("  nodes", A.shape[0], "edges", edge_count, "features", X.shape, "classes", classes)
    print("  sample X", np.round(X[:3], 3).tolist())


In [ ]:

results = []

for item in ladder:
    A, X, d, common, views = build_graph_tensors(item["G"], item["X"])
    Z = np.column_stack([X, d / max(1.0, float(d.max()))])
    pred = centroid_predict(Z, item["y"])
    metric = node_accuracy(pred, item["y"])
    results.append({"rung": item["rung"], "G": item["G"], "pred": pred, "metric": metric})
    print(f'{item["rung"]} {item["name"]:24s} node-accuracy={metric:.3f}')


In [ ]:

plot_graph_panels(results, "node accuracy")


## 7. Pitfall on the hardest rung: missing is not negative

On D5, every zero in $A$ is tempting to treat as a negative edge. That creates a huge and often false negative class. The fix is to evaluate with a masked set of observed positives and sampled held-out negatives.

In [ ]:

hard = ladder[-1]
A = hard["A"]
n = A.shape[0]
positive_pairs = int(A.sum() // 2)
all_pairs = n * (n - 1) // 2
unobserved_pairs = all_pairs - positive_pairs
wrong_negative_ratio = unobserved_pairs / max(1, positive_pairs)
masked_negative_count = positive_pairs
fixed_negative_ratio = masked_negative_count / positive_pairs

print("all unobserved as negatives ratio", round(wrong_negative_ratio, 2))
print("masked held-out negatives ratio", round(fixed_negative_ratio, 2))


## 8. Evaluate it + Practice

- Report the ladder metric (node accuracy) next to a no-skill majority-class or random partition baseline.
- Sanity-check D1 against the hand-computed assertions before trusting larger rungs.
- Ablation: remove degree from the representation and watch hub-heavy D5 change.
- Failure signals: accuracy high on D1 but unstable on D5, or a link loss dominated by unobserved zeros.
- Keep all experiments CPU-only and seeded.

Practice prompts:
1. Change D3 noise and predict which metric moves first.
2. Add a small bridge between communities and inspect the visualization.
3. Replace the readout with a majority baseline and compare the curve.